In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("salting").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/04 19:50:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType

schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("timestamp",TimestampType(), False),
    StructField("user_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("campaign_id", StringType(), False),
    StructField("device_type", StringType(), True),
    StructField("revenue_usd", DoubleType(), True)
])

df = spark.read.schema(schema=schema).json('data/raw')

In [5]:
df.show(10)

+--------------------+--------------------+--------------------+----------+-----------+-----------+-----------+
|            event_id|           timestamp|             user_id|event_type|campaign_id|device_type|revenue_usd|
+--------------------+--------------------+--------------------+----------+-----------+-----------+-----------+
|aef631bc-5907-46a...|2026-10-11 01:45:...|7041c3a6-2658-4cd...|impression|   camp_003|     mobile|     0.0042|
|5dc5d2cd-051b-440...|2026-09-27 18:21:...|f5ced9b7-8d71-4d9...|impression|   camp_003|     mobile|     0.0021|
|c17a65e6-63bc-4bd...|2026-09-30 17:38:...|85d23e59-fcc9-4f9...|impression|   camp_001|     tablet|     0.0057|
|fb8e5035-4b2e-4ab...|2026-10-25 09:24:...|11b302c9-bc2b-42c...|impression|   camp_002|     mobile|     0.0095|
|f777a2b4-86c0-485...|2026-10-13 11:36:...|64848b41-9ca5-4f8...|impression|   camp_884|     mobile|     0.0026|
|2195f6e9-ac5b-435...|2026-10-04 01:19:...|b9fd8768-1f7d-4a9...|impression|   camp_002|     mobile|     

## Data skew 

In [6]:
import pyspark.sql.functions as F

df_raw = df.withColumn("partition", F.spark_partition_id())

In [7]:
df_raw.show(10)

+--------------------+--------------------+--------------------+----------+-----------+-----------+-----------+---------+
|            event_id|           timestamp|             user_id|event_type|campaign_id|device_type|revenue_usd|partition|
+--------------------+--------------------+--------------------+----------+-----------+-----------+-----------+---------+
|aef631bc-5907-46a...|2026-10-11 01:45:...|7041c3a6-2658-4cd...|impression|   camp_003|     mobile|     0.0042|        0|
|5dc5d2cd-051b-440...|2026-09-27 18:21:...|f5ced9b7-8d71-4d9...|impression|   camp_003|     mobile|     0.0021|        0|
|c17a65e6-63bc-4bd...|2026-09-30 17:38:...|85d23e59-fcc9-4f9...|impression|   camp_001|     tablet|     0.0057|        0|
|fb8e5035-4b2e-4ab...|2026-10-25 09:24:...|11b302c9-bc2b-42c...|impression|   camp_002|     mobile|     0.0095|        0|
|f777a2b4-86c0-485...|2026-10-13 11:36:...|64848b41-9ca5-4f8...|impression|   camp_884|     mobile|     0.0026|        0|
|2195f6e9-ac5b-435...|20

In [8]:
df_raw.groupBy(F.col("partition")).\
    agg(F.count(F.col("partition")).alias("count")).\
    orderBy(F.desc(F.col("count"))).\
    show()

+---------+------+
|partition| count|
+---------+------+
|        0|400000|
|        1|400000|
|        2|400000|
|        3|400000|
|        4|400000|
|        5|400000|
|        6|400000|
|        7|400000|
|        8|400000|
|        9|400000|
|       10|400000|
|       11|400000|
|       12|200000|
+---------+------+



In [9]:
df_raw.groupBy(F.col("campaign_id")).\
        agg(F.sum(F.col("revenue_usd")).alias("revenue")).\
        orderBy(F.desc("revenue"),).\
        show()

+-----------+------------------+
|campaign_id|           revenue|
+-----------+------------------+
|   camp_001| 3598050.845899729|
|   camp_002|3585685.6591997314|
|   camp_003|3553278.8851997284|
|   camp_846|1127.3041999999998|
|   camp_636|1109.5220000000002|
|   camp_997|         1063.5078|
|   camp_013|1029.4326999999998|
|   camp_310|         1023.8345|
|   camp_883| 998.8388999999999|
|   camp_587|          987.1684|
|   camp_766| 985.8603999999999|
|   camp_511|          978.5304|
|   camp_912| 975.8302999999999|
|   camp_485|          962.9868|
|   camp_147|          962.8918|
|   camp_615| 961.0623000000002|
|   camp_326|          960.2083|
|   camp_179| 948.0996999999999|
|   camp_390|          946.4269|
|   camp_629|          945.3812|
+-----------+------------------+
only showing top 20 rows


In [10]:
# To see what happened exactly and make the skew visible, we will do the shuffle ourselves 

df_shuffle = df.repartition(13, F.col("campaign_id"))
df_shuffle = df_shuffle.withColumn("partition", F.spark_partition_id())

df_shuffle.groupBy(F.col("partition")).\
    agg(F.count(F.col("partition")).alias("count")).\
    orderBy(F.desc(F.col("count"))).\
    show()

+---------+-------+
|partition|  count|
+---------+-------+
|        6|1602943|
|        2|1600169|
|        0|1599207|
|       11|  21506|
|        1|  21403|
|        3|  21077|
|        7|  21044|
|        8|  20960|
|       10|  20075|
|       12|  19090|
|        9|  18378|
|        5|  17566|
|        4|  16582|
+---------+-------+



In [11]:
df_shuffle.groupBy("campaign_id"). \
        agg(F.sum("revenue_usd").alias("revenue")). \
        orderBy(F.desc("revenue")).show()

+-----------+------------------+
|campaign_id|           revenue|
+-----------+------------------+
|   camp_001|3598050.8458983884|
|   camp_002| 3585685.659198417|
|   camp_003|3553278.8851985615|
|   camp_846|1127.3041999999998|
|   camp_636|1109.5219999999997|
|   camp_997|1063.5078000000003|
|   camp_013|1029.4326999999998|
|   camp_310|1023.8345000000002|
|   camp_883| 998.8388999999995|
|   camp_587| 987.1684000000002|
|   camp_766| 985.8604000000003|
|   camp_511| 978.5304000000001|
|   camp_912| 975.8302999999994|
|   camp_485| 962.9868000000002|
|   camp_147| 962.8917999999993|
|   camp_615| 961.0622999999996|
|   camp_326| 960.2083000000001|
|   camp_179| 948.0997000000001|
|   camp_390| 946.4269000000006|
|   camp_629|          945.3812|
+-----------+------------------+
only showing top 20 rows


## Fix data skew with salting

In [12]:
# 1 row ~= 150 bytes
salt_nbr = int(((1_600_000*150)*1e-6)/128)
salt_nbr

1

In [13]:
df_raw.withColumn("salt", (F.rand()*salt_nbr).cast("int")).\
        groupBy(F.col("campaign_id"), F.col("salt")). \
        agg(F.sum("revenue_usd").alias("revenue")). \
        groupBy(F.col("campaign_id")). \
        agg(F.sum("revenue").alias("revenue")). \
        orderBy(F.desc("revenue")).\
        show()

+-----------+------------------+
|campaign_id|           revenue|
+-----------+------------------+
|   camp_001| 3598050.845899729|
|   camp_002|3585685.6591997314|
|   camp_003|3553278.8851997284|
|   camp_846|1127.3041999999998|
|   camp_636|1109.5220000000002|
|   camp_997|         1063.5078|
|   camp_013|1029.4326999999998|
|   camp_310|         1023.8345|
|   camp_883| 998.8388999999999|
|   camp_587|          987.1684|
|   camp_766| 985.8603999999999|
|   camp_511|          978.5304|
|   camp_912| 975.8302999999999|
|   camp_485|          962.9868|
|   camp_147|          962.8918|
|   camp_615| 961.0623000000002|
|   camp_326|          960.2083|
|   camp_179| 948.0996999999999|
|   camp_390|          946.4269|
|   camp_629|          945.3812|
+-----------+------------------+
only showing top 20 rows


## Join with campaign_metadata small table using salting 

In [14]:
# Generate 1000 mock campaigns
metadata_records = [(f"camp_{i:03d}", f"Promo_Alpha_{i}", "Active") for i in range(1, 1001)]

metadata_schema = StructType([
    StructField("campaign_id", StringType(), False),
    StructField("campaign_name", StringType(), False),
    StructField("status", StringType(), False)
])

df_metadata = spark.createDataFrame(metadata_records, metadata_schema)

In [23]:
import math
# 1 row ~= 150 bytes
# 1 MB = 1024**2 bytes
salt_nbr = math.ceil(((1_600_000*150)/(1024**2))/128)
salt_nbr

2

In [24]:
df_raw = df_raw.withColumn("salt", (F.rand() * salt_nbr).cast("int"))

In [25]:
df_metadata = df_metadata.withColumn("salt_array", F.array([F.lit(i) for i in range(salt_nbr)])). \
                withColumn("salt", F.explode("salt_array"))
df_metadata.show()

+-----------+--------------+------+----------+----+
|campaign_id| campaign_name|status|salt_array|salt|
+-----------+--------------+------+----------+----+
|   camp_001| Promo_Alpha_1|Active|    [0, 1]|   0|
|   camp_001| Promo_Alpha_1|Active|    [0, 1]|   1|
|   camp_002| Promo_Alpha_2|Active|    [0, 1]|   0|
|   camp_002| Promo_Alpha_2|Active|    [0, 1]|   1|
|   camp_003| Promo_Alpha_3|Active|    [0, 1]|   0|
|   camp_003| Promo_Alpha_3|Active|    [0, 1]|   1|
|   camp_004| Promo_Alpha_4|Active|    [0, 1]|   0|
|   camp_004| Promo_Alpha_4|Active|    [0, 1]|   1|
|   camp_005| Promo_Alpha_5|Active|    [0, 1]|   0|
|   camp_005| Promo_Alpha_5|Active|    [0, 1]|   1|
|   camp_006| Promo_Alpha_6|Active|    [0, 1]|   0|
|   camp_006| Promo_Alpha_6|Active|    [0, 1]|   1|
|   camp_007| Promo_Alpha_7|Active|    [0, 1]|   0|
|   camp_007| Promo_Alpha_7|Active|    [0, 1]|   1|
|   camp_008| Promo_Alpha_8|Active|    [0, 1]|   0|
|   camp_008| Promo_Alpha_8|Active|    [0, 1]|   1|
|   camp_009

In [28]:
df_joined = df_raw.join(df_metadata, ["campaign_id", "salt"], 'inner')

In [29]:
df_joined.show()

+-----------+----+--------------------+--------------------+--------------------+----------+-----------+-----------+---------+---------------+------+----------+
|campaign_id|salt|            event_id|           timestamp|             user_id|event_type|device_type|revenue_usd|partition|  campaign_name|status|salt_array|
+-----------+----+--------------------+--------------------+--------------------+----------+-----------+-----------+---------+---------------+------+----------+
|   camp_656|   0|3a2d8a88-9f8a-483...|2026-10-02 10:41:...|e0e4f41b-abb6-4de...|  purchase|     mobile|     57.439|        0|Promo_Alpha_656|Active|    [0, 1]|
|   camp_521|   1|4b6d3009-56f6-4be...|2026-10-18 22:55:...|fc6c0ab6-7179-4d4...|  purchase|     mobile|    32.2014|        0|Promo_Alpha_521|Active|    [0, 1]|
|   camp_981|   0|24e7866a-48b4-48e...|2026-10-24 08:09:...|bfff9863-6c11-43f...|impression|    desktop|     0.0089|        0|Promo_Alpha_981|Active|    [0, 1]|
|   camp_182|   0|11af229c-a18d-46

In [34]:
df_joined = df_joined.drop(F.col("salt"), F.col("salt_array"))

df_joined.show(5)

+-----------+--------------------+--------------------+--------------------+----------+-----------+-----------+---------+---------------+------+
|campaign_id|            event_id|           timestamp|             user_id|event_type|device_type|revenue_usd|partition|  campaign_name|status|
+-----------+--------------------+--------------------+--------------------+----------+-----------+-----------+---------+---------------+------+
|   camp_656|3a2d8a88-9f8a-483...|2026-10-02 10:41:...|e0e4f41b-abb6-4de...|  purchase|     mobile|     57.439|        0|Promo_Alpha_656|Active|
|   camp_521|4b6d3009-56f6-4be...|2026-10-18 22:55:...|fc6c0ab6-7179-4d4...|  purchase|     mobile|    32.2014|        0|Promo_Alpha_521|Active|
|   camp_981|24e7866a-48b4-48e...|2026-10-24 08:09:...|bfff9863-6c11-43f...|impression|    desktop|     0.0089|        0|Promo_Alpha_981|Active|
|   camp_182|11af229c-a18d-465...|2026-10-04 08:13:...|e4216336-6b1c-432...|impression|     mobile|     0.0074|        0|Promo_Alp

## Solve data skew using broadcast join

In [37]:
df_joined_brd = df_raw.join(F.broadcast(df_metadata),
                            on="campaign_id", 
                            how='inner')

df_joined_brd.show(5)

+-----------+--------------------+--------------------+--------------------+----------+-----------+-----------+---------+----+-------------+------+----------+----+
|campaign_id|            event_id|           timestamp|             user_id|event_type|device_type|revenue_usd|partition|salt|campaign_name|status|salt_array|salt|
+-----------+--------------------+--------------------+--------------------+----------+-----------+-----------+---------+----+-------------+------+----------+----+
|   camp_003|aef631bc-5907-46a...|2026-10-11 01:45:...|7041c3a6-2658-4cd...|impression|     mobile|     0.0042|        0|   1|Promo_Alpha_3|Active|    [0, 1]|   1|
|   camp_003|aef631bc-5907-46a...|2026-10-11 01:45:...|7041c3a6-2658-4cd...|impression|     mobile|     0.0042|        0|   1|Promo_Alpha_3|Active|    [0, 1]|   0|
|   camp_003|5dc5d2cd-051b-440...|2026-09-27 18:21:...|f5ced9b7-8d71-4d9...|impression|     mobile|     0.0021|        0|   1|Promo_Alpha_3|Active|    [0, 1]|   1|
|   camp_003|5dc